In [181]:
print("DAY-2 Workshop on AI-Driven RNA Therapeutics: From Data to Drug Design")

DAY-2 Workshop on AI-Driven RNA Therapeutics: From Data to Drug Design


## feature Engineering for RNA sequence

In [182]:
from Bio.Seq import Seq
from  Bio.SeqUtils import gc_fraction

In [183]:
import pandas as pd
data={
  "sequence":[("AUCGAGCGC"),("GGCAAAAA"),("CAGGCA"),("AUGCCA")],
 
  "efficacy":[.7,.8,.2,.4]
}
df=pd.DataFrame(data)
df

,sequence,efficacy
0,AUCGAGCGC,0.7
1,GGCAAAAA,0.8
2,CAGGCA,0.2
3,AUGCCA,0.4


In [184]:
import numpy as np
mapping={
  "A":[1,0,0,0],
  "C":[0,1,0,0],
  "G":[0,0,1,0],
  "U":[0,0,0,1]
}
def one_hot_encoding(seq):
  return np.array([mapping[n]for n in seq] ).flatten()

In [185]:
df["one_hot_sequence "]=df["sequence"].apply(one_hot_encoding)
df

,sequence,efficacy,one_hot_sequence
0,AUCGAGCGC,0.7,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, ..."
1,GGCAAAAA,0.8,"[0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, ..."
2,CAGGCA,0.2,"[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, ..."
3,AUGCCA,0.4,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, ..."


In [186]:
def gc_content(sequence):
    seq = Seq(sequence)
    return gc_fraction(seq) * 100

In [187]:
def gc_skew(sequence):
    seq = Seq(sequence)
    g = seq.count("G")
    c = seq.count("C")
    return (g - c) / (g + c) if (g + c) > 0 else 0

def au_skew(sequence):
    seq = Seq(sequence)
    a = seq.count("A")
    u = seq.count("U") or seq.count("T")  # handles both RNA and DNA
    return (a - u) / (a + u) if (a + u) > 0 else 0

In [188]:
df["gc_content "]=df["sequence"].apply(gc_content)
df["gc_skew"]=df["sequence"].apply(gc_skew)
df["au_skew"]=df["sequence"].apply(au_skew)
df

,sequence,efficacy,one_hot_sequence,gc_content,gc_skew,au_skew
0,AUCGAGCGC,0.7,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, ...",66.666667,0.000000,0.333333
1,GGCAAAAA,0.8,"[0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, ...",37.500000,0.333333,1.000000
2,CAGGCA,0.2,"[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, ...",66.666667,0.000000,1.000000
3,AUGCCA,0.4,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, ...",50.000000,-0.333333,0.333333


## siRNA Efficiency Prediction Pipeline

In [189]:
hu=pd.read_csv("./Data/Hu.csv")
hu.head()

,siRNA,mRNA,label,y,td
0,CUAAUAUGUUAAUUGAUUU,AAAUUGAAAGGAAUUGUAUAAAUCAAUUAACAUAUUAGCUGAGUUG...,0.344519,0,"-1.6,-2.08,-10.48,0,0,-149.37999999999997,0.52..."
1,AAUAUGUUAAUUGAUUUAU,UAAAAUUGAAAGGAAUUGUAUAAAUCAAUUAACAUAUUAGCUGAGU...,0.286353,0,"0.17,-0.93,-6.82,0,0,-144.55999999999997,0.526..."
2,GAUUUAUACAAUUCCUUUC,CUUAUUUUUAGAUAAAAUUGAAAGGAAUUGUAUAAAUCAAUUAACA...,0.383296,0,"0.0,-2.35,-12.44,0,1,-163.86,0.473684210526315..."
3,CAAUUCCUUUCAAUUUUAU,UUGGUCUGCUUAUUUUUAGAUAAAAUUGAAAGGAAUUGUAUAAAUC...,0.271439,0,"-1.46,-2.11,-10.44,0,0,-152.68999999999994,0.5..."
4,CAGACCAAAAUUAAAUAAG,UGGAAUCUUAUGUAACUUUCUUAUUUAAUUUUGGUCUGCUUAUUUU...,0.389262,0,"-0.03,-2.11,-10.44,0,0,-157.33999999999997,0.1..."


In [190]:
import pandas as pd 
from collections import Counter

In [191]:
#load the data
def load_data(path):
  df=pd.read_csv(path)
  df=df.dropna(subset=["efficacy"])
  return df
  

In [192]:
#load the data
patent_ENsiRNA=load_data("./Data/patent_ENsiRNA.csv")
patent_ENsiRNA.head()


,siRNA,anti seq,sense seq,mRNA_seq,position,efficacy
0,AD-59453,AACAACAGAGATTTGCCTG,CAGGCAAATCTCTGTTGTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,401,0.888
1,AD-59395,TTTCTCATCAATTTTTTTC,GAAAAAAATTGATGAGAAA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,948,0.873
2,AD-59477,AAGAGTGCGGCATCTTTCC,GGAAAGATGCCGCACTCTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1241,0.855
3,AD-59492,TCTTGAAGAAGATGAGACA,TGTCTCATCTTCTTCAAGA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,881,0.852
4,AD-59361,ATTGCTTGCACGTAGATGT,ACATCTACGTGCAAGCAAT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1991,0.849


In [ ]:
#drop siRNA because it just index
patent_ENsiRNA.drop("siRNA",axis=1,inplace=True)
patent_ENsiRNA.head()
patent_ENsiRNA["anti_seq"]=patent_ENsiRNA["anti seq"].str.upper()
patent_ENsiRNA["sense_seq"]=patent_ENsiRNA["sense seq"].str.upper()
patent_ENsiRNA["mRNA_seq"]=patent_ENsiRNA["mRNA_seq"].str.upper()


In [197]:
patent_ENsiRNA.head()

,anti seq,sense seq,mRNA_seq,position,efficacy,anti_seq,sense_seq
0,AACAACAGAGATTTGCCTG,CAGGCAAATCTCTGTTGTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,401,0.888,AACAACAGAGATTTGCCTG,CAGGCAAATCTCTGTTGTT
1,TTTCTCATCAATTTTTTTC,GAAAAAAATTGATGAGAAA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,948,0.873,TTTCTCATCAATTTTTTTC,GAAAAAAATTGATGAGAAA
2,AAGAGTGCGGCATCTTTCC,GGAAAGATGCCGCACTCTT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1241,0.855,AAGAGTGCGGCATCTTTCC,GGAAAGATGCCGCACTCTT
3,TCTTGAAGAAGATGAGACA,TGTCTCATCTTCTTCAAGA,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,881,0.852,TCTTGAAGAAGATGAGACA,TGTCTCATCTTCTTCAAGA
4,ATTGCTTGCACGTAGATGT,ACATCTACGTGCAAGCAAT,CTGTATATTAAGGCGCCGGCGATCGCGGCCTGAGGCTGCTCCCGGA...,1991,0.849,ATTGCTTGCACGTAGATGT,ACATCTACGTGCAAGCAAT


## Feature engineering


In [194]:
mapping={
  "A":[1,0,0,0],
  "C":[0,1,0,0],
  "G":[0,0,1,0],
  "U":[0,0,0,1],
  "T":[0,0,0,1]
}
def one_hot_encoding(seq):
  return np.array([mapping[n]for n in seq] ).flatten()

In [195]:
def gc_content(sequence):
    seq = Seq(sequence)
    return gc_fraction(seq) * 100

def gc_skew(sequence):
    seq = Seq(sequence)
    g = seq.count("G")
    c = seq.count("C")
    return (g - c) / (g + c) if (g + c) > 0 else 0

def au_skew(sequence):
    seq = Seq(sequence)
    a = seq.count("A")
    u = seq.count("U") or seq.count("T")  # handles both RNA and DNA
    return (a - u) / (a + u) if (a + u) > 0 else 0  

In [196]:
df["gc_content "]=df["sequence"].apply(gc_content)
df["gc_skew"]=df["sequence"].apply(gc_skew)
df["au_skew"]=df["sequence"].apply(au_skew)
df

,sequence,efficacy,one_hot_sequence,gc_content,gc_skew,au_skew
0,AUCGAGCGC,0.7,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, ...",66.666667,0.000000,0.333333
1,GGCAAAAA,0.8,"[0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, ...",37.500000,0.333333,1.000000
2,CAGGCA,0.2,"[0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, ...",66.666667,0.000000,1.000000
3,AUGCCA,0.4,"[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, ...",50.000000,-0.333333,0.333333


In [ ]:
x=[]
y=